# ﻿# ================================================================

DATATHON - PASSOS MÁGICOS

PARTE 4 - MACHINE LEARNING

MODELO PREDITIVO DE RISCO DE DEFASAGEM

## 1. IMPORTAÇÃO DAS BIBLIOTECAS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from google.colab import files

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate
)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

## 2. UPLOAD DA BASE

In [ ]:
print("=" * 70)
print("UPLOAD DA BASE DE DADOS")
print("=" * 70)

print("\nSelecione o arquivo Excel do Datathon.\n")

uploaded = files.upload()

arquivos_excel = [
    nome
    for nome in uploaded.keys()
    if nome.lower().endswith((".xlsx", ".xls"))
]

if not arquivos_excel:
    raise ValueError("Nenhum arquivo Excel foi enviado.")

arquivo = arquivos_excel[0]

print("\nArquivo carregado:")
print(arquivo)

## 3. LEITURA DAS BASES DE 2023 E 2024

In [ ]:
df23 = pd.read_excel(
    arquivo,
    sheet_name="PEDE2023"
)

df24 = pd.read_excel(
    arquivo,
    sheet_name="PEDE2024"
)

print("\n" + "=" * 70)
print("BASES CARREGADAS")
print("=" * 70)

print(f"Registros em 2023: {len(df23)}")
print(f"Registros em 2024: {len(df24)}")

## 4. CRUZAMENTO DOS ALUNOS

In [ ]:
# Utilizamos o RA para identificar o mesmo aluno
# em 2023 e 2024.
#
# A ideia é utilizar os indicadores de 2023
# para prever a situação do aluno em 2024.

base = df23.merge(
    df24[["RA", "Defasagem"]],
    on="RA",
    how="inner",
    suffixes=("_2023", "_2024")
)

base = base[
    base["Defasagem_2024"].notna()
].copy()

print("\n" + "=" * 70)
print("CRUZAMENTO DAS BASES")
print("=" * 70)

print(
    f"Alunos presentes em 2023 e 2024: {len(base)}"
)

## 5. DEFINIÇÃO DO TARGET

In [ ]:
# 1 = aluno em risco
# 0 = aluno sem risco
#
# Consideramos risco quando a Defasagem em 2024
# apresenta valor menor que zero.

base["risco_2024"] = (
    base["Defasagem_2024"] < 0
).astype(int)

## 6. DISTRIBUIÇÃO DO TARGET

In [ ]:
sem_risco = (
    base["risco_2024"] == 0
).sum()

com_risco = (
    base["risco_2024"] == 1
).sum()

percentual_sem_risco = (
    base["risco_2024"] == 0
).mean()

percentual_com_risco = (
    base["risco_2024"] == 1
).mean()

print("\n" + "=" * 70)
print("DISTRIBUIÇÃO DO TARGET")
print("=" * 70)

print(
    f"Sem risco: {sem_risco} "
    f"({percentual_sem_risco:.1%})"
)

print(
    f"Em risco: {com_risco} "
    f"({percentual_com_risco:.1%})"
)

## 7. FEATURES

In [ ]:
# Utilizamos somente informações disponíveis em 2023
# para prever a situação em 2024.

features = [
    "Idade",
    "Ano ingresso",
    "IAA",
    "IEG",
    "IPS",
    "IPP",
    "IDA",
    "IPV",
    "IAN",
    "INDE 2023",
    "Defasagem_2023"
]

X = base[features].copy()

y = base["risco_2024"].copy()

## 8. CONVERSÃO PARA NUMÉRICO

In [ ]:
for coluna in features:

    X[coluna] = pd.to_numeric(
        X[coluna],
        errors="coerce"
    )

print("\n" + "=" * 70)
print("FEATURES UTILIZADAS")
print("=" * 70)

for feature in features:
    print("-", feature)

print(
    f"\nQuantidade de alunos: {X.shape[0]}"
)

print(
    f"Quantidade de features: {X.shape[1]}"
)

## 9. VALORES AUSENTES

In [ ]:
print("\n" + "=" * 70)
print("VALORES AUSENTES")
print("=" * 70)

faltantes = X.isnull().sum()

faltantes = faltantes[
    faltantes > 0
]

if len(faltantes) > 0:

    print(faltantes)

    print(
        "\nOs valores ausentes serão preenchidos "
        "com a mediana durante o treinamento."
    )

else:

    print(
        "Não existem valores ausentes."
    )

## 10. DIVISÃO ENTRE TREINO E TESTE

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y
)

print("\n" + "=" * 70)
print("DIVISÃO DOS DADOS")
print("=" * 70)

print(
    f"Treinamento: {len(X_train)} alunos"
)

print(
    f"Teste: {len(X_test)} alunos"
)

## 11. REGRESSÃO LOGÍSTICA

In [ ]:
modelo_logistico = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    ),

    (
        "scaler",
        StandardScaler()
    ),

    (
        "model",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )

])

## 12. ÁRVORE DE DECISÃO

In [ ]:
modelo_arvore = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    ),

    (
        "model",
        DecisionTreeClassifier(
            max_depth=5,
            min_samples_leaf=10,
            class_weight="balanced",
            random_state=42
        )
    )

])

## 13. RANDOM FOREST

In [ ]:
modelo_random_forest = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    ),

    (
        "model",
        RandomForestClassifier(

            n_estimators=500,

            max_depth=8,

            min_samples_leaf=4,

            class_weight="balanced",

            random_state=42,

            n_jobs=-1
        )
    )

])

## 14. MODELOS

In [ ]:
modelos = {

    "Regressão Logística":
        modelo_logistico,

    "Árvore de Decisão":
        modelo_arvore,

    "Random Forest":
        modelo_random_forest

}

## 15. TREINAMENTO E AVALIAÇÃO

In [ ]:
resultados = []

modelos_treinados = {}

print("\n" + "=" * 70)
print("TREINAMENTO DOS MODELOS")
print("=" * 70)

for nome, modelo in modelos.items():

    print("\n")
    print("-" * 70)
    print(nome.upper())
    print("-" * 70)

    # Treinamento
    modelo.fit(
        X_train,
        y_train
    )

    # Previsão das classes
    y_pred = modelo.predict(
        X_test
    )

    # Probabilidade de risco
    y_prob = modelo.predict_proba(
        X_test
    )[:, 1]

    # Métricas
    acuracia = accuracy_score(
        y_test,
        y_pred
    )

    precisao = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        y_prob
    )

    resultados.append({

        "Modelo": nome,

        "Acurácia": acuracia,

        "Precisão": precisao,

        "Recall": recall,

        "F1-score": f1,

        "ROC-AUC": roc_auc

    })

    modelos_treinados[
        nome
    ] = modelo

    print(
        f"Acurácia: {acuracia:.2%}"
    )

    print(
        f"Precisão: {precisao:.2%}"
    )

    print(
        f"Recall: {recall:.2%}"
    )

    print(
        f"F1-score: {f1:.2%}"
    )

    print(
        f"ROC-AUC: {roc_auc:.2%}"
    )

## 16. COMPARAÇÃO DOS MODELOS

In [ ]:
resultado_df = pd.DataFrame(
    resultados
)

resultado_df = resultado_df.sort_values(
    "ROC-AUC",
    ascending=False
).reset_index(drop=True)

resultado_visual = resultado_df.copy()

for coluna in [
    "Acurácia",
    "Precisão",
    "Recall",
    "F1-score",
    "ROC-AUC"
]:

    resultado_visual[coluna] = (
        resultado_visual[coluna] * 100
    ).round(2).astype(str) + "%"

print("\n")
print("=" * 70)
print("COMPARAÇÃO FINAL DOS MODELOS")
print("=" * 70)

display(
    resultado_visual
)

## 17. SELEÇÃO DO MELHOR MODELO

In [ ]:
melhor_nome = resultado_df.iloc[0][
    "Modelo"
]

melhor_modelo = modelos_treinados[
    melhor_nome
]

print("\n" + "=" * 70)
print("MELHOR MODELO")
print("=" * 70)

print(
    melhor_nome
)

## 18. PREDIÇÃO DO MELHOR MODELO

In [ ]:
y_pred_melhor = melhor_modelo.predict(
    X_test
)

y_prob_melhor = melhor_modelo.predict_proba(
    X_test
)[:, 1]

## 19. MATRIZ DE CONFUSÃO

In [ ]:
matriz = confusion_matrix(
    y_test,
    y_pred_melhor
)

print("\n" + "=" * 70)
print("MATRIZ DE CONFUSÃO")
print("=" * 70)

print(matriz)

print("\nInterpretação:")

print(
    f"Verdadeiros negativos: {matriz[0][0]}"
)

print(
    f"Falsos positivos: {matriz[0][1]}"
)

print(
    f"Falsos negativos: {matriz[1][0]}"
)

print(
    f"Verdadeiros positivos: {matriz[1][1]}"
)

## 20. GRÁFICO DA MATRIZ DE CONFUSÃO

In [ ]:
ConfusionMatrixDisplay(

    confusion_matrix=matriz,

    display_labels=[
        "Sem risco",
        "Risco"
    ]

).plot()

plt.title(
    f"Matriz de Confusão - {melhor_nome}"
)

plt.show()

## 21. RELATÓRIO DE CLASSIFICAÇÃO

In [ ]:
print("\n" + "=" * 70)
print("RELATÓRIO DE CLASSIFICAÇÃO")
print("=" * 70)

print(

    classification_report(

        y_test,

        y_pred_melhor,

        target_names=[
            "Sem risco",
            "Risco"
        ],

        zero_division=0

    )

)

## 22. CURVA ROC

In [ ]:
RocCurveDisplay.from_estimator(

    melhor_modelo,

    X_test,

    y_test

)

plt.title(
    f"Curva ROC - {melhor_nome}"
)

plt.show()

## 23. VALIDAÇÃO CRUZADA

In [ ]:
cv = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=42

)

resultado_cv = cross_validate(

    modelos[melhor_nome],

    X,

    y,

    cv=cv,

    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc"
    ],

    n_jobs=-1

)

cv_acuracia = (
    resultado_cv[
        "test_accuracy"
    ].mean()
)

cv_precisao = (
    resultado_cv[
        "test_precision"
    ].mean()
)

cv_recall = (
    resultado_cv[
        "test_recall"
    ].mean()
)

cv_f1 = (
    resultado_cv[
        "test_f1"
    ].mean()
)

cv_roc_auc = (
    resultado_cv[
        "test_roc_auc"
    ].mean()
)

print("\n" + "=" * 70)
print("VALIDAÇÃO CRUZADA - 5 FOLDS")
print("=" * 70)

print(
    f"Acurácia média: {cv_acuracia:.2%}"
)

print(
    f"Precisão média: {cv_precisao:.2%}"
)

print(
    f"Recall médio: {cv_recall:.2%}"
)

print(
    f"F1-score médio: {cv_f1:.2%}"
)

print(
    f"ROC-AUC médio: {cv_roc_auc:.2%}"
)

## 24. TREINAMENTO DO MODELO FINAL

In [ ]:
modelo_final = modelos[
    melhor_nome
]

modelo_final.fit(
    X,
    y
)

print("\n" + "=" * 70)
print("MODELO FINAL TREINADO")
print("=" * 70)

print(
    f"Modelo: {melhor_nome}"
)

print(
    f"Registros utilizados: {len(X)}"
)

## 25. FEATURE IMPORTANCE

In [ ]:
print("\n" + "=" * 70)
print("FEATURE IMPORTANCE")
print("=" * 70)

# A Random Forest e a Árvore de Decisão possuem
# o atributo feature_importances_.
#
# Caso a Regressão Logística seja selecionada,
# utilizamos o valor absoluto dos coeficientes.

estimador_final = modelo_final.named_steps[
    "model"
]

if hasattr(
    estimador_final,
    "feature_importances_"
):

    importancias = (
        estimador_final.feature_importances_
    )

    tipo_importancia = (
        "Feature Importance do modelo"
    )

elif hasattr(
    estimador_final,
    "coef_"
):

    importancias = np.abs(
        estimador_final.coef_[0]
    )

    tipo_importancia = (
        "Importância baseada nos coeficientes"
    )

else:

    importancias = np.zeros(
        len(features)
    )

    tipo_importancia = (
        "Importância não disponível"
    )

## 26. TABELA DE FEATURE IMPORTANCE

In [ ]:
feature_importance_df = pd.DataFrame({

    "Feature": features,

    "Importância": importancias

})

feature_importance_df = (
    feature_importance_df
    .sort_values(
        "Importância",
        ascending=False
    )
    .reset_index(drop=True)
)

# Percentual relativo

feature_importance_df[
    "Importância (%)"
] = (

    feature_importance_df[
        "Importância"
    ]

    /

    feature_importance_df[
        "Importância"
    ].sum()

    * 100

)

print(
    tipo_importancia
)

print()

display(
    feature_importance_df
)

## 27. GRÁFICO DE FEATURE IMPORTANCE

In [ ]:
grafico_importancia = (
    feature_importance_df
    .sort_values(
        "Importância",
        ascending=True
    )
)

plt.figure(
    figsize=(10, 7)
)

plt.barh(

    grafico_importancia[
        "Feature"
    ],

    grafico_importancia[
        "Importância"
    ]

)

plt.xlabel(
    "Importância"
)

plt.ylabel(
    "Indicador"
)

plt.title(
    f"Feature Importance - {melhor_nome}"
)

plt.tight_layout()

plt.show()

## 28. TOP 5 INDICADORES MAIS IMPORTANTES

In [ ]:
top5 = feature_importance_df.head(
    5
)

print("\n" + "=" * 70)
print("TOP 5 INDICADORES MAIS IMPORTANTES")
print("=" * 70)

for indice, linha in top5.iterrows():

    print(

        f"{indice + 1}. "
        f"{linha['Feature']} - "
        f"{linha['Importância (%)']:.2f}%"

    )

## 29. SALVAR FEATURE IMPORTANCE EM CSV

In [ ]:
feature_importance_df.to_csv(

    "feature_importance.csv",

    index=False,

    encoding="utf-8-sig"

)

print(
    "\nArquivo criado: feature_importance.csv"
)

## 30. SALVAR MODELO

In [ ]:
joblib.dump(

    modelo_final,

    "modelo_risco_defasagem.pkl"

)

joblib.dump(

    features,

    "features_modelo.pkl"

)

print("\n" + "=" * 70)
print("ARQUIVOS GERADOS")
print("=" * 70)

print(
    "modelo_risco_defasagem.pkl"
)

print(
    "features_modelo.pkl"
)

print(
    "feature_importance.csv"
)

## 31. EXEMPLO DE PREVISÃO

In [ ]:
aluno_exemplo = X_test.iloc[
    [0]
]

probabilidade_risco = (
    modelo_final.predict_proba(
        aluno_exemplo
    )[0][1]
)

previsao = (
    modelo_final.predict(
        aluno_exemplo
    )[0]
)

print("\n" + "=" * 70)
print("EXEMPLO DE PREVISÃO")
print("=" * 70)

print(
    f"Probabilidade de risco: "
    f"{probabilidade_risco:.2%}"
)

if previsao == 1:

    print(
        "Classificação: ALUNO EM RISCO"
    )

else:

    print(
        "Classificação: ALUNO SEM RISCO"
    )

## 32. RESUMO FINAL

In [ ]:
resultado_melhor = resultado_df.iloc[
    0
]

principal_feature = (
    feature_importance_df.iloc[0][
        "Feature"
    ]
)

segunda_feature = (
    feature_importance_df.iloc[1][
        "Feature"
    ]
)

terceira_feature = (
    feature_importance_df.iloc[2][
        "Feature"
    ]
)

print("\n")
print("=" * 70)
print("RESUMO FINAL")
print("=" * 70)

print(
    f"""
Foram analisados {len(base)} alunos presentes
nas bases de 2023 e 2024.

A situação observada foi:

Sem risco:
{sem_risco} ({percentual_sem_risco:.1%})

Em risco:
{com_risco} ({percentual_com_risco:.1%})

Foram avaliados três algoritmos:

1. Regressão Logística
2. Árvore de Decisão
3. Random Forest

O modelo com maior ROC-AUC foi:

{melhor_nome}

RESULTADOS NO CONJUNTO DE TESTE

Acurácia:
{resultado_melhor['Acurácia']:.2%}

Precisão:
{resultado_melhor['Precisão']:.2%}

Recall:
{resultado_melhor['Recall']:.2%}

F1-score:
{resultado_melhor['F1-score']:.2%}

ROC-AUC:
{resultado_melhor['ROC-AUC']:.2%}

VALIDAÇÃO CRUZADA

Acurácia média:
{cv_acuracia:.2%}

Precisão média:
{cv_precisao:.2%}

Recall médio:
{cv_recall:.2%}

F1-score médio:
{cv_f1:.2%}

ROC-AUC médio:
{cv_roc_auc:.2%}

FEATURE IMPORTANCE

Os três indicadores com maior importância
para as previsões do modelo foram:

1. {principal_feature}
2. {segunda_feature}
3. {terceira_feature}

IMPORTANTE:

A Feature Importance indica quais variáveis
foram mais utilizadas pelo modelo para realizar
suas previsões.

Ela NÃO significa que esses indicadores sejam
necessariamente a causa da defasagem.
"""
)

## 33. CONCLUSÃO

In [ ]:
print("\n" + "=" * 70)
print("CONCLUSÃO")
print("=" * 70)

print(
    f"""
Foi desenvolvido um modelo de Machine Learning para
identificar antecipadamente alunos com risco de defasagem.

Para evitar o uso de informações futuras, foram utilizados
os indicadores disponíveis em 2023 para prever a situação
observada em 2024.

O modelo selecionado foi:

{melhor_nome}

No conjunto de teste, apresentou ROC-AUC de
{resultado_melhor['ROC-AUC']:.2%} e Recall de
{resultado_melhor['Recall']:.2%}.

Na validação cruzada com 5 folds, apresentou
ROC-AUC médio de {cv_roc_auc:.2%}.

A análise de Feature Importance também permitiu
identificar quais indicadores tiveram maior influência
nas decisões realizadas pelo modelo.

Os três indicadores de maior importância foram:

1. {principal_feature}
2. {segunda_feature}
3. {terceira_feature}

Esses resultados podem auxiliar a Associação Passos Mágicos
na identificação antecipada de alunos que necessitam de
maior acompanhamento.

A importância das variáveis representa associação
com as previsões do modelo e não deve ser interpretada
como relação causal.
"""
)

## 34. DOWNLOAD DOS ARQUIVOS

In [ ]:
print("\n" + "=" * 70)
print("DOWNLOAD")
print("=" * 70)

print(
    "\nSerão baixados três arquivos:"
)

print(
    "1. modelo_risco_defasagem.pkl"
)

print(
    "2. features_modelo.pkl"
)

print(
    "3. feature_importance.csv"
)

files.download(
    "modelo_risco_defasagem.pkl"
)

files.download(
    "features_modelo.pkl"
)

files.download(
    "feature_importance.csv"
)